# Phiên Kaggle: train M3 (EXP-004) + M2 (EXP-003) trong **một lần chạy**

Một clone, một lần build dataset, hai mô hình. Thứ tự **M3 trước, M2 sau** —
đây là quyết định phụ thuộc chứ không phải ngẫu nhiên:

| | Thí nghiệm | Mô hình | Thời gian (T4) | Kết quả |
| :--- | :--- | :--- | ---: | :--- |
| 1 | EXP-004 | M3 ASTNN (RvNN, 428k tham số) | **~35 phút** | Có sớm, dễ backup |
| 2 | EXP-003 | M2 GraphCodeBERT (126M tham số) | **~7,8h** | Dài, dễ chết |

Nếu session chết ở giữa cell M2 thì **M3 vẫn còn nguyên** và đã đóng gói sẵn. Nếu
đảo ngược thứ tự, 8h của M2 có thể bay mà chưa có M3.

Tổng ~8,4h < giới hạn 12h của Kaggle. Mỗi mô hình có `--time-budget-s` riêng để
dừng graceful, sau đó vẫn chạy evaluate.

**Setup 1 lần:** Settings → Accelerator = **GPU T2 ×2 hoặc P100/T4**, Internet = **ON**.

**Sửa 1 chỗ:** cell 2 — dòng `REPO`. Chạy cell theo thứ tự, **đừng Run All**.

### Điều kiện so sánh được (quan trọng)
EXP-002 (đã xong) / EXP-003 (M2) dùng **chung** recipe: `max_length 256`, 3 epoch,
batch 16, accum 2, chỉ khác `hf_id` (DEC-016, DEC-020, DEC-022). EXP-004 là mô hình
khác hẳn về họ nên **không** buộc giống hyperparameter (đã ghi trong config) —
khi so sánh phải nói rõ điểm này, không so số F1 trần.

In [ ]:
import json
import os
import subprocess
import sys

# Cách 1 (khuyên dùng): Kaggle Add-ons → Secrets → GITHUB_TOKEN, HF_TOKEN, attach vào notebook
GITHUB_TOKEN = ""  # GitHub PAT — bắt buộc nếu repo PRIVATE
HF_TOKEN = ""  # HuggingFace token — tải checkpoint ổn định
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret("GITHUB_TOKEN")
    HF_TOKEN = HF_TOKEN or _s.get_secret("HF_TOKEN")
except Exception as e:
    print("dùng token dán tay:", e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("GITHUB_TOKEN:", "OK" if GITHUB_TOKEN else "TRỐNG", "| HF_TOKEN:", "OK" if HF_TOKEN else "TRỐNG")


In [ ]:
REPO = "kichirou244/SemanticCodeCloneDetection"
assert REPO != "YOUR_USER/YOUR_REPO", "Sửa REPO trước khi chạy cell tiếp theo"

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f"$ {cmd}\n{r.stdout}{r.stderr}")
    if r.returncode != 0:
        raise RuntimeError(f"lệnh thất bại (rc={r.returncode}): {cmd}")
    return r.stdout

def py(code, **kw):
    """Chạy snippet Python một dòng (không dùng newline trong code)."""
    return sh(f'{sys.executable} -c {json.dumps(code)}', **kw)

# Idempotent: chạy lại cell này sẽ pull bản mới thay vì clone lần nữa.
if os.path.isdir("project"):
    print("project đã tồn tại → pull thay vì clone")
    sh("git -C project pull --ff-only")
else:
    url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git" if GITHUB_TOKEN else f"https://github.com/{REPO}.git"
    sh(f'git clone "{url}" project')
sh(f"git -C project remote set-url origin https://github.com/{REPO}.git")  # xóa token khỏi remote
sh("git -C project log --oneline -1")
os.chdir("project")
print("cwd =", os.getcwd())
leak = GITHUB_TOKEN and GITHUB_TOKEN in sh("git remote -v")
print("còn sót token trong remote?", "CÓ — xóa notebook!" if leak else "KHÔNG")

# EXP-003 phải dùng CHUNG runner với EXP-002 — assert để chắc chắn
from src.experiments import exp002_codebert as e2, exp003_graphcodebert as e3
assert set(e2.__all__) == set(e3.__all__), "wrapper lệch public API"
print("EXP-002/EXP-003 dùng CHUNG runner:", e2.train_model is e3.train_model)


In [ ]:
# KHÔNG đụng torch: Kaggle đã có bản CUDA tương thích, `pip install transformers`
# vô tình kéo torch về bản CPU/default sẽ làm train rất chậm (hoặc chết).
import importlib

# tree-sitter-cpp BẮT BUỘC cho EXP-004 (ASTNN build cache AST từ nó);
# EXP-002/003 không cần nhưng dùng chung môi trường nên cài luôn.
need = []
for mod, pkg in [("transformers", "transformers"), ("datasets", "datasets"),
                 ("sklearn", "scikit-learn"), ("yaml", "pyyaml"),
                 ("tqdm", "tqdm"), ("numpy", "numpy"),
                 ("tree_sitter", "tree-sitter"), ("tree_sitter_cpp", "tree-sitter-cpp")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
print("cần cài:", need or "không")
if need:
    sh("pip install -q " + " ".join(need))

import torch, transformers, datasets, sklearn, numpy
assert torch.cuda.is_available(), "Kaggle chưa gắn GPU — vào Settings → Accelerator"
print({"torch": torch.__version__, "cuda": torch.version.cuda, "transformers": transformers.__version__,
       "gpu": torch.cuda.get_device_name(0),
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory/1e9, 1),
       "cpu_cores": os.cpu_count()})
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
# EXP-004 build AST cache bằng tree-sitter; kiểm tra parser C++ nạp được
py("from src.representations.ast.extract import get_parser; t = get_parser('cpp').parse(b'int f(){return 1;}'); print('tree-sitter cpp OK, nodes =', t.root_node.descendant_count)")


In [ ]:
# Rebuild dataset foundation POJ-104 MỘT LẦN cho cả hai mô hình.
# data/processed + data/splits bị .gitignore nên không có sẵn trong clone — phải build.
sh("python scripts/build_dataset.py --config configs/pairs/poj104.yaml")
py("import json; m = json.load(open('data/manifests/poj104.json')); "
   "print('leakage_passed =', m['leakage_passed']); "
   "print('pairs =', {k: v for k, v in m.get('pairs', {}).items()})")
sh("python scripts/env_report.py --output results/exp003_graphcodebert_poj104/env.json")
sh("python scripts/env_report.py --output results/exp004_astnn_poj104/env.json")
sh("df -h /kaggle/working | tail -1")
# Cùng sha256 manifest/pairs là điều kiện để M1/M2/M3 so sánh được
sh("sha256sum data/manifests/poj104.json")


In [ ]:
# Preflight cả hai: GPU/VRAM/config trước khi tốn 8h. Phải IN ra báo cáo.
sh("python -m src.experiments.exp004_astnn --config configs/exp004_astnn.yaml "
   "--mode preflight --device cuda")
sh("python -m src.experiments.exp003_graphcodebert --config configs/exp003_graphcodebert.yaml "
   "--mode preflight --device cuda")


## Bước 1 — M3 ASTNN (EXP-004), khoảng 35 phút

Không tokenize, không truncate: giữ toàn bộ cây statement (DEC-021). GRU 200 đơn vị,
428k tham số, fp32. `diagnostic` in số optimizer step đo thật để ngoại suy thời gian
trên T4 — phải PASS trước khi train.

In [ ]:
sh("python -m src.experiments.exp004_astnn --config configs/exp004_astnn.yaml "
   "--mode diagnostic --device cuda")


In [ ]:
# Train M3 + evaluate (train_model tự gọi evaluate khi hết epoch/budget).
# Cell đầu tiên sẽ build cache AST cho toàn bộ split (~62k code) — vài phút.
MODE = "train"  # "train" | "resume"
TIME_BUDGET_S = 45 * 60  # rộng hơn nhiều so với ~35 phút dự kiến
sh(f"python -m src.experiments.exp004_astnn --config configs/exp004_astnn.yaml "
   f"--mode {MODE} --device cuda --time-budget-s {TIME_BUDGET_S}")


In [ ]:
# Đóng gói M3 ngay, trước khi dồn 8h vào M2.
# Checkpoint của M3 rất nhỏ (~5MB) nhưng vẫn rút bản nhẹ cho đồng nhất quy trình:
# đường tải về + `--mode evaluate` local là một đường duy nhất cho mọi mô hình.
OUT3 = "results/exp004_astnn_poj104"
sh(f"test -f {OUT3}/EVAL_COMPLETE.json && echo 'M3 EVAL COMPLETE' || echo 'M3 EVAL CHUA XONG'")
py(f"from pathlib import Path; from src.experiments.transformer_pair_runner import export_eval_weights; export_eval_weights(Path('{OUT3}'))")
# cache AST nằm ở artifacts/ast_cache (ngoài thư mục result) nên không cần exclude.
sh(f"tar --exclude='*checkpoint.pt' --exclude='*.tmp' -czf exp004_results.tar.gz {OUT3}")
sh("ls -lh exp004_results.tar.gz")
sh(f"ls -lh {OUT3}/")
print(sh(f"cat {OUT3}/metrics.json"))


### ⏸ DỪNG Ở ĐÂY — backup M3 trước khi chạy 8h M2

Bấm **Save Version** (hoặc **Save Output**) ngay bây giờ để `exp004_results.tar.gz`
nằm an toàn ngoài phiên. Nếu M2 làm session chết, bạn vẫn còn kết quả M3 mà không
phải train lại 35 phút.

Sau khi lưu, chạy tiếp các cell M2 bên dưới.

## Bước 2 — M2 GraphCodeBERT (EXP-003), khoảng 7,8h

Cùng code path với EXP-002 (đã xong), chỉ khác `hf_id`. Cell này là cell dài nhất
của phiên — nếu bị kill, `last_checkpoint.pt` được flush mỗi epoch (DEC-018).

**Resume nếu session chết:** chép `results/exp003_graphcodebert_poj104/last_checkpoint.pt`
từ version đã save vào đúng thư mục, rồi đặt `MODE = "resume"` và chạy lại cell.

In [ ]:
sh("python -m src.experiments.exp003_graphcodebert --config configs/exp003_graphcodebert.yaml "
   "--mode diagnostic --device cuda")


In [ ]:
# FULL TRAINING M2 — ~7,8h, đừng chạy Run All.
MODE = "train"  # "train" | "resume"
TIME_BUDGET_S = 9 * 3600  # dừng graceful trước giới hạn 12h của Kaggle
sh(f"python -m src.experiments.exp003_graphcodebert --config configs/exp003_graphcodebert.yaml "
   f"--mode {MODE} --device cuda --time-budget-s {TIME_BUDGET_S}")


In [ ]:
# Đóng gói M2: chỉ giữ checkpoint BẢN NHẸ (~0,5GB) để chạy evaluate local.
OUT2 = "results/exp003_graphcodebert_poj104"
sh(f"test -f {OUT2}/EVAL_COMPLETE.json && echo 'M2 EVAL COMPLETE' || echo 'M2 EVAL CHUA XONG'")
# EXP-004 không re-export (wrapper này không có __all__), nên cả hai đều gọi hàm
# của runner chung — cùng một đường code cho mọi mô hình.
py(f"from pathlib import Path; from src.experiments.transformer_pair_runner import export_eval_weights; export_eval_weights(Path('{OUT2}'))")
sh(f"tar --exclude='*checkpoint.pt' --exclude='*.tmp' -czf exp003_results.tar.gz {OUT2}")
sh("ls -lh exp003_results.tar.gz")
sh(f"ls -lh {OUT2}/")
print(sh(f"cat {OUT2}/metrics.json"))


### Sau khi tải 2 file tar về

| File | Nội dung | Kích thước |
| :--- | :--- | ---: |
| `exp004_results.tar.gz` | M3 ASTNN: metrics, predictions, `best_weights_only.pt` | vài MB |
| `exp003_results.tar.gz` | M2 GraphCodeBERT: metrics, predictions, `best_weights_only.pt` | ~0,5GB |

Giải nén **đè** lên `results/exp004_astnn_poj104/` và
`results/exp003_graphcodebert_poj104/`, rồi chạy evaluate **local** (GPU 8GB trở lên,
mỗi lần ~5 phút cho M2, ~1 phút cho M3):

```bash
python -m src.experiments.exp003_graphcodebert --config configs/exp003_graphcodebert.yaml --mode evaluate --device cuda
python -m src.experiments.exp004_astnn      --config configs/exp004_astnn.yaml      --mode evaluate --device cuda
```

`evaluate_model` ưu tiên `best_checkpoint.pt` nếu có, không thì tự dùng
`best_weights_only.pt` — nên **không** cần tải checkpoint 1,44GB của M2.

### Cập nhật docs sau khi có số liệu (repo yêu cầu code + docs tiến song song)

1. M2: `reports/012_graphcodebert_results.md` — test metrics + so sánh M1 (DEC-016),
   nêu rõ khác biệt chỉ do pretrained weights.
2. M3: `reports/013_astnn_results.md` — test metrics + so sánh, nêu rõ M3 **không
   truncate** nên thuận lợi là do biểu diễn chứ không phải do "thông minh hơn".
3. `docs/experiment_registry.md` + `configs/*.yaml`: `experiment.status` → `COMPLETED`.
4. So sánh tổng hợp M1/M2/M3 trên cùng split ở `docs/experiment_protocol.md` hoặc
   report tổng hợp — dùng `predictions/*_scores.jsonl` để tính lại mọi số.

### EXP-005 (M4 CPG-GNN) chưa chạy được trong phiên này
Cần Joern (chưa cài, cần JRE) để dựng CPG cho ~62k file C/C++ — đo tương đối với
tree-sitter (3.153 file/s, đo trên POJ-104) thì Joern chậm hơn 1–2 đơn vị, tức vài
giờ đến hàng chục giờ, và chưa có sẵn model CPG-GNN. Chi tiết: `reports/005`.